# Лекция 11. Деревья и кучи

Дерево — обычный способ представить вложенность: категории расходов, папки, подразделения, пункты меню. Куча решает другую практическую задачу: быстро находить следующий самый важный элемент. Разберём обе структуры на коротких примерах без скрытой магии. Хотя двоичную кучу удобно рисовать деревом, в Python она хранится в обычном списке.

## Цели

После лекции вы сможете:

- называть корень, родителя, ребёнка, лист, глубину, высоту и поддерево;
- писать рекурсивный обход с базовым случаем;
- выбирать DFS или BFS и реализовывать их через стек или очередь;
- поддерживать инвариант бинарного дерева поиска;
- честно оценивать сложность BST через высоту дерева;
- объяснять устройство двоичной кучи в списке;
- пользоваться min-heap и новым max-heap API Python 3.14;
- строить стабильную очередь с приоритетом и находить top-k без полной сортировки.

## Перед началом

Нужны списки, классы данных, стек и очередь из прошлых занятий. План неспешного рассказа: 20 минут на устройство дерева и рекурсию, 20 минут на DFS/BFS, 20 минут на BST, 20 минут на кучи и приоритетные очереди, 10 минут на неожиданные случаи, самопроверку и вопросы. Код короткий: основное время уйдёт на проговаривание инвариантов и трассировку нескольких шагов.

## Дерево как вложенная структура

Возьмём категории расходов. У верхней категории `all` нет родителя — это **корень**. У `food` есть дети `groceries` и `cafes`. Узел без детей называется **листом**. Все узлы ниже выбранного узла образуют его **поддерево**.

**Глубина** узла — число рёбер от корня до него. **Высота** узла — длина самого длинного пути от него до листа. Поэтому глубина относится к положению сверху, а высота — к оставшейся части снизу. Высота всего дерева равна высоте корня.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Category:
    name: str
    amount: int = 0
    children: list["Category"] = field(default_factory=list)

expenses = Category("all", children=[
    Category("food", children=[
        Category("groceries", 3200),
        Category("cafes", 1800),
    ]),
    Category("transport", 900),
])

В этой модели каждый узел, кроме корня, достижим ровно одним путём сверху. Это и отличает дерево от общего графа. Если один и тот же объект добавить в несколько ветвей или сослаться из ребёнка обратно на родителя, получится граф; наивный рекурсивный обход сможет посчитать объект дважды или уйти по циклу. Поэтому алгоритм опирается не только на поля класса, но и на обещание о форме данных.

## Рекурсия: решить задачу для узла и его поддеревьев

Рекурсивная функция вызывает саму себя на меньших частях задачи. Для суммы категории правило буквально повторяет определение: взять сумму текущего узла и прибавить суммы всех дочерних поддеревьев.

Базовый случай здесь не обязательно записывать отдельным `if`. У листа список детей пуст, поэтому `sum(...)` возвращает ноль и дальнейших вызовов нет. Важно уметь назвать, почему процесс заканчивается: каждый вызов спускается на один уровень, а дерево конечно.

In [ ]:
def total_amount(node: Category) -> int:
    return node.amount + sum(total_amount(child) for child in node.children)

assert total_amount(expenses) == 5900

Каждый незавершённый вызов хранится в стеке вызовов. Для дерева из `N` узлов функция работает за `O(N)`: каждый узел посещён один раз. Дополнительная память — `O(H)`, где `H` — высота, потому что одновременно активен один путь от корня вниз.

CPython ограничивает глубину рекурсии, чтобы защитить стек интерпретатора. Поэтому рекурсия естественна для неглубокой иерархии категорий, но цепочку из многих тысяч узлов безопаснее обходить собственным стеком. Повышать лимит вслепую — не исправление структуры алгоритма.

## DFS: сначала в глубину

Depth-first search идёт по выбранной ветви до конца и только затем возвращается к соседям. Он удобен, когда нужно обработать всё поддерево, найти любой подходящий путь или сохранить память на широком дереве.

В preorder-варианте текущий узел обрабатывается до детей. В postorder — после детей; именно postorder естественен, когда результат родителя зависит от готовых результатов потомков. Inorder имеет особый смысл для бинарного дерева поиска и появится ниже.

In [ ]:
def preorder_names(node: Category) -> list[str]:
    result = [node.name]
    for child in node.children:
        result.extend(preorder_names(child))
    return result

assert preorder_names(expenses) == [
    "all", "food", "groceries", "cafes", "transport"
]

Рекурсию можно сделать явной: положить корень в стек, извлекать последний узел и добавлять его детей. Поскольку стек работает LIFO, детей кладут в обратном порядке, если хотят сохранить привычный обход слева направо.

Собственный стек не делает алгоритм асимптотически быстрее, но убирает зависимость от лимита рекурсии и позволяет хранить вместе с узлом дополнительное состояние.

In [ ]:
def preorder_iterative(root: Category) -> list[str]:
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.name)
        stack.extend(reversed(node.children))
    return result

assert preorder_iterative(expenses) == preorder_names(expenses)

## BFS: слой за слоем

Breadth-first search сначала посещает все узлы глубины 0, затем глубины 1 и так далее. Для этого нужна FIFO-очередь. В Python используем `collections.deque`: извлечение слева через `popleft()` занимает `O(1)`, тогда как `list.pop(0)` сдвигает оставшиеся элементы.

BFS на невзвешенном графе первым находит путь с наименьшим числом рёбер. В дереве это означает ближайший по глубине подходящий узел. Цена — очередь может одновременно хранить почти целый широкий слой.

In [ ]:
from collections import deque

def names_by_level(root: Category) -> list[tuple[str, int]]:
    result = []
    queue = deque([(root, 0)])
    while queue:
        node, depth = queue.popleft()
        result.append((node.name, depth))
        for child in node.children:
            queue.append((child, depth + 1))
    return result

assert names_by_level(expenses) == [
    ("all", 0), ("food", 1), ("transport", 1),
    ("groceries", 2), ("cafes", 2),
]

## Как выбрать обход

- Нужен любой результат глубоко в ветви, агрегация поддерева или postorder — обычно DFS.
- Нужен ближайший результат или обработка по уровням — BFS.
- Дерево очень глубокое — итеративный DFS вместо рекурсивного.
- Дерево очень широкое — DFS часто хранит меньше узлов, чем BFS.

Оба обхода посещают каждый достижимый узел и работают за `O(N)`. Разница не в большой O по времени, а в порядке результата и максимуме одновременно сохранённых узлов.

## Бинарное дерево поиска

В бинарном дереве у узла не больше двух детей. В **бинарном дереве поиска** дополнительно действует инвариант: все меньшие ключи находятся слева, все большие — справа. Это правило относится ко всему поддереву, а не только к непосредственным детям.

Ниже ключом будет сумма платежа, а значением — его описание. Для простоты повторный ключ обновляет значение.

In [ ]:
@dataclass
class SearchNode:
    key: int
    value: str
    left: "SearchNode | None" = None
    right: "SearchNode | None" = None

In [ ]:
def bst_insert(root: SearchNode | None, key: int, value: str) -> SearchNode:
    if root is None:
        return SearchNode(key, value)
    node = root
    while True:
        if key == node.key:
            node.value = value
            return root
        if key < node.key:
            if node.left is None:
                node.left = SearchNode(key, value)
                return root
            node = node.left
        else:
            if node.right is None:
                node.right = SearchNode(key, value)
                return root
            node = node.right

def bst_find(root: SearchNode | None, key: int) -> str | None:
    node = root
    while node is not None:
        if key == node.key:
            return node.value
        node = node.left if key < node.key else node.right
    return None

Поиск и вставка идут по одному пути, поэтому стоят `O(H)`, где `H` — высота. У сбалансированного дерева `H = O(log N)`. Но наш класс не балансирует дерево: если вставить уже отсортированные ключи, каждый новый узел окажется справа и высота станет `N - 1`. Тогда операция займёт `O(N)`.

Для обычного поиска по точному ключу встроенный `dict` почти всегда проще и быстрее. BST интересно, когда нужен упорядоченный обход, диапазоны или понимание структуры, на которой строятся более сложные сбалансированные деревья.

Inorder-обход идёт `левое поддерево → узел → правое поддерево`. Благодаря инварианту BST ключи получаются по возрастанию. Само бинарное дерево без инварианта поиска такой гарантии не даёт.

In [ ]:
def bst_items(root: SearchNode | None) -> list[tuple[int, str]]:
    if root is None:
        return []
    return bst_items(root.left) + [(root.key, root.value)] + bst_items(root.right)

root = None
for key, value in [(30, "taxi"), (10, "coffee"), (20, "lunch"), (50, "hotel")]:
    root = bst_insert(root, key, value)

assert bst_find(root, 20) == "lunch"
assert [key for key, _ in bst_items(root)] == [10, 20, 30, 50]

## Двоичная куча: экстремум наверху

Куча решает не поиск произвольного ключа, а быстрое получение минимума или максимума. В min-heap каждый родитель не больше детей; в max-heap — не меньше. Между соседними ветвями порядка нет, поэтому куча не является отсортированным списком и не является BST.

Двоичная куча — почти полное дерево: уровни заполняются слева направо. Поэтому ссылки на детей не нужны. Для индекса `i` левый ребёнок лежит в `2*i + 1`, правый — в `2*i + 2`, родитель — в `(i - 1) // 2`.

In [ ]:
import heapq

amounts = [900, 120, 1500, 450, 700]
heapq.heapify(amounts)
assert amounts[0] == 120
heapq.heappush(amounts, 300)
assert heapq.heappop(amounts) == 120
assert amounts[0] == 300

`heapify` перестраивает существующий список на месте за `O(N)`. Последовательные `N` вызовов `heappush` стоили бы `O(N log N)`. Чтение корня — `O(1)`, добавление и извлечение корня — `O(log N)`, потому что элемент поднимается или опускается максимум на высоту кучи.

После `heappop` новым корнем становится следующий минимум. Остальная часть списка гарантирует только отношение родителя и детей; печатать её как готовый рейтинг нельзя.

## Max-heap без отрицательных приоритетов

> **Новое в Python 3.14.** В `heapq` появились симметричные функции `heapify_max`, `heappush_max`, `heappop_max`, `heappushpop_max` и `heapreplace_max`. Раньше max-heap обычно изображали min-heap с отрицательными числами. Старый приём продолжает встречаться в коде и остаётся полезным для поддержки старых версий, но в нашем курсе целевая версия — 3.14.

In [ ]:
from heapq import heapify_max, heappop_max, heappush_max

priorities = [2, 5, 1, 4]
heapify_max(priorities)
assert priorities[0] == 5
heappush_max(priorities, 10)
assert heappop_max(priorities) == 10

## Стабильная очередь с приоритетом

У обращения есть приоритет, но одинаково срочные обращения надо обслуживать в порядке поступления. Кортежи сравниваются слева направо, поэтому добавим монотонный номер. В max-heap запись `(priority, -order, request_id)` сначала выбирает больший приоритет, а при равенстве — большее `-order`, то есть более ранний номер.

Счётчик также не позволяет сравнению дойти до сложного объекта задачи. Это важно: два словаря нельзя упорядочить через `<`.

In [ ]:
def serve_requests(requests: list[dict]) -> list[str]:
    heap = []
    for order, request in enumerate(requests):
        heappush_max(heap, (request["priority"], -order, request["id"]))
    return [heappop_max(heap)[2] for _ in range(len(heap))]

requests = [
    {"id": "r-1", "priority": 2},
    {"id": "r-2", "priority": 5},
    {"id": "r-3", "priority": 5},
]
assert serve_requests(requests) == ["r-2", "r-3", "r-1"]

## Top-k без полной сортировки

Если из миллиона операций нужны десять крупнейших, сортировать миллион значений за `O(N log N)` избыточно. Держим min-heap размера не больше `k`: корень — худший из текущих победителей. Новый элемент заменяет корень, только если лучше него. Получаем `O(N log k)` времени и `O(k)` дополнительной памяти.

Для разовой задачи есть `heapq.nlargest(k, data, key=...)`. Ручная реализация полезна, когда данные приходят потоком или вместе с рейтингом нужно поддерживать свои правила стабильности.

In [ ]:
def top_amounts(amounts: list[int], k: int) -> list[int]:
    if k == 0:
        return []
    heap: list[int] = []
    for amount in amounts:
        if len(heap) < k:
            heapq.heappush(heap, amount)
        elif amount > heap[0]:
            heapq.heapreplace(heap, amount)
    return sorted(heap, reverse=True)

assert top_amounts([120, 900, 450, 1500, 700], 3) == [1500, 900, 700]

## Обновление и отмена: ленивое удаление

Куча быстро удаляет только корень. Искать произвольную задачу внутри списка и восстанавливать инвариант неудобно. Практический планировщик обычно хранит словарь актуальных записей, а старую запись при обновлении помечает удалённой. `pop` пропускает помеченные вершины, пока не встретит живую.

Физический список может быть длиннее числа живых задач — это ожидаемая плата за быстрые обновления. Если устаревших записей стало слишком много, кучу можно периодически собрать заново через `heapify_max`.

## Неожиданно, но по правилам

### 1. Куча не отсортирована

После `heapify` гарантирован только экстремум в корне и отношения родителей с детьми. Например, `[1, 4, 2, 9, 7]` — корректная min-heap, хотя `4 > 2`. Полный порядок потребовал бы другой работы.

### 2. `heapify` изменяет список и возвращает `None`

Стандартные изменяющие операции контейнеров обычно не возвращают сам контейнер. Поэтому `heap = heapq.heapify(values)` уничтожит ссылку на список в переменной `heap`: там окажется `None`.

### 3. Итеративный DFS легко меняет порядок

Если положить детей в стек слева направо, первым извлечётся правый. `reversed(children)` нужен не для корректности достижения узлов, а для совпадения порядка с рекурсивным обходом.

### 4. BST может стать связным списком

Последовательная вставка `1, 2, 3, 4` в несбалансированное BST создаёт только правых детей. Правило BST соблюдено полностью, но обещанного на словах `O(log N)` нет.

### 5. Равный приоритет может привести к `TypeError`

Если записи имеют вид `(priority, task_dict)`, при равных приоритетах Python попробует сравнить словари. Уникальный числовой счётчик между приоритетом и задачей предотвращает это.

### 6. Старый трюк max-heap меняет знак

До Python 3.14 приоритет `10` клали в min-heap как `-10`. Ошибка при одном из двух преобразований давала правдоподобный, но обратный порядок. Новый max-heap API хранит исходный смысл числа явно.

In [ ]:
values = [4, 1, 7, 2]
returned = heapq.heapify(values)
assert returned is None
assert values[0] == 1
assert values != sorted(values)

## Самопроверка

1. Чем глубина узла отличается от его высоты?
2. Почему рекурсивная сумма завершается на листе без отдельного `if`?
3. Что хранится в памяти при рекурсивном DFS?
4. Зачем разворачивать детей перед добавлением в стек?
5. Когда BFS лучше DFS?
6. Почему `deque.popleft()` предпочтительнее `list.pop(0)`?
7. Как звучит инвариант BST?
8. Почему сложность поиска в нашем BST равна `O(H)`, а не всегда `O(log N)`?
9. Почему inorder BST даёт отсортированные ключи?
10. Чем инвариант кучи отличается от полной сортировки?
11. Какие функции max-heap появились в Python 3.14?
12. Зачем очереди с приоритетом монотонный счётчик?
13. Когда top-k выгоднее полной сортировки?
14. Зачем при отмене задачи нужно ленивое удаление?

## Источники

- [Документация `heapq`](https://docs.python.org/3/library/heapq.html) — min-heap, новый max-heap API Python 3.14, приоритетные очереди и top-k.
- [Документация `collections.deque`](https://docs.python.org/3/library/collections.html#collections.deque) — очередь с быстрыми операциями с обоих концов.
- [Лимит рекурсии CPython](https://docs.python.org/3/library/sys.html#sys.getrecursionlimit) — зачем глубокой рекурсии нужна осторожность.

Документация подтверждает интерфейсы; оценки сложности выводим из числа посещённых узлов и высоты структуры.

## Итоги

- Дерево представляет иерархию, а рекурсия повторяет её вложенное устройство.
- DFS идёт по ветви, BFS — по уровням; стек и очередь делают порядок явным.
- BST хранит глобальный порядок поддеревьев, но без балансировки может выродиться.
- Куча хранит только достаточный порядок для быстрого экстремума и реализуется списком.
- Python 3.14 добавил прямой max-heap API в `heapq`; старый код часто использует отрицательные приоритеты.
- Счётчик делает очередь с приоритетом стабильной и не даёт сравнивать сами задачи.
- Куча размера `k` находит крупнейшие элементы за `O(N log k)` без полной сортировки.

На семинаре вручную протрассируем обходы, соберём BST и две реальные очереди с приоритетом.